In [ ]:
import time
import geopandas as gpd
import os
import json
import pandas as pd
import shapely
import matplotlib.pyplot as plt
import os
import requests

In [ ]:
os.environ['PL_API_KEY']='YOUR API KEY'
# pass in your API key

PLANET_API_KEY = os.getenv('PL_API_KEY')
# Setup the API Key from the `PL_API_KEY` environment variable

BASE_URL = "https://api.planet.com/data/v1"

session = requests.Session()
#setup a session

session.auth = (PLANET_API_KEY, "")
#authenticate session with user name and password, pass in an empty string for the password

res = session.get(BASE_URL)
#make a get request to the Data API

print(res.status_code)
# test response

print(res.text)
# print response body

arctic_region = [[
        [-180, 60],
        [180, 60],
        [180, 90],
        [-180, 90],
        [-180, 60]
]]

query_data = {
    "item_types": ["PSScene"],
    "filter": {
        "type": "AndFilter",
        "config": [
            {
                "type": "DateRangeFilter",
                "field_name": "acquired",
                "config": {
                    "gte": "2023-03-01T00:00:00.000Z",
                    "lte": "2023-04-01T00:00:00.000Z"
                }
            },
            {
                "type": "GeometryFilter",
                "field_name": "geometry",
                "config": {
                    "type": "Polygon",
                    "coordinates": arctic_region, #ice_boundary_json['features'][0]['geometry']['coordinates']
                }
            },
        ]
    }
}

res = session.post('https://api.planet.com/data/v1/quick-search', json=query_data)
print("Status Code for the Quick Search query", res.status_code)
res = res.json()
len(res['features'])
search_results = []
count = 0
# print(res['features'])
while True:
    search_result = gpd.GeoDataFrame(res['features'])
    # print(len(search_result))
    if len(search_result) == 0:
        break
    search_result.geometry = search_result['geometry'].apply(lambda geom: shapely.geometry.shape(geom))
    #Original code: search_result.geometry = search_result.geometry.apply(lambda geom: shapely.geometry.shape(geom))

    # expand the properties column (json) into a dataframe
    properties_df = search_result.properties.apply(lambda x: pd.Series(x))
    search_result = pd.concat([search_result, properties_df], axis=1)
    # parse the acquired time
    search_result['capture_time'] = search_result['acquired']
    search_result['capture_time'] = pd.to_datetime(search_result.capture_time, format='ISO8601')
    search_result['capture_date'] = search_result.capture_time.apply(lambda x: x.date())
    search_result['capture_month'] = search_result.capture_time.apply(lambda x: x.month)
    search_result['capture_year'] = search_result.capture_time.apply(lambda x: x.year)
    search_result['capture_day'] = search_result.capture_time.apply(lambda x: x.day)
    search_result['capture_hour'] = search_result.capture_time.apply(lambda x: x.hour)
    search_result['capture_minute'] = search_result.capture_time.apply(lambda x: x.minute)
    search_result['capture_second'] = search_result.capture_time.apply(lambda x: x.second)

    search_results.append(search_result)

    if '_next' not in res['_links']:
        break

    try:
        res_ = session.get(res['_links']['_next'])
    except:
        print("Error in getting the next page")
        time.sleep(10)
        continue

    # with open(f'/content/drive/MyDrive/ST_intersection/planet_2022.txt', 'a') as f:
    #     f.write(f"{count:05d}: " + res['_links']['_next'] + "\n")

    if res_.status_code != 200:
        print("Status Code for the Quick Search query", res_.status_code)
        time.sleep(10)
        continue
    res = res_.json()


if len(search_results) != 0:
    search_results = pd.concat(search_results)
    search_results.crs = 'EPSG:4326'
    sr_shp = search_results
    sr_shp[search_results.select_dtypes(exclude=['number', 'geometry']).columns] = sr_shp[search_results.select_dtypes(exclude=['number', 'geometry']).columns].astype(str)

    sr_shp.to_file('/content/drive/MyDrive/ST_intersection/planet_2023/planet2023_3.geojson', driver='GeoJSON')                      #<<<<<<<<<<<----------- Change of year only
    pd.DataFrame(sr_shp.assign(geometry=sr_shp["geometry"].apply(lambda p: p.wkt))).to_csv('/content/drive/MyDrive/ST_intersection/planet_2023/planet2023_3.csv')  #<<<<<<<<<<------- Change of year only

    print("Number of rows within the box:  " + str(len(search_results)))
    # search_result.to_file(f'/content/drive/MyDrive/ST_intersection/search_result_{count:05d}.geojson', driver='GeoJSON')
    count += 1